# Santa 2026: 177 experiments, what worked and what did not

This notebook is a write-up of five days of work on the Elf Calculator: every line of attack we tried, with the numbers, including
the many that failed. It ends with a validated submission that scores **128,524**. I have experimented to use Claude Code with 0 code edits myself to see how far I can take a CC only solution.
The pace in which it can absorb new information (from public notebooks including public puzzles), generate new ideas itself and/or validate any ideas I wanted to test was insanely fast.

Hope you can incorporate some learnings to empower your own pipeline / weaknesses as I have built from others. This competition is just getting started - look forward to what new tricks / ideas we can generate.

Contents:

1. How we think about the problem
2. Score over time
3. What worked, in order of how much it gave
4. What did not work
5. Insights worth more than any single trick
6. The submission (decoded, validated against `puzzles.csv`, written to `submission.csv`)
7. How this was made: the whole solution was built in Claude Code

A note on provenance: our file is a per-puzzle merge. Whenever a public notebook had a shorter program for a puzzle we took it,
and the search described below ran on top. The last section attributes every merged program to the notebook and author it
came from.

## 1. How we think about the problem

- **The score is the number of distinct values a program computes.** No good program computes a value twice, so the task is:
  find the smallest set of values that contains every target and in which each value is one instruction away from two earlier
  ones, then order it so that at most 48 values are alive at once.
- **A program is a value graph, and all savings come from sharing.** In a good program for 30 random 50-bit targets, 14% of the
  values are targets, 44% are used once and 42% are used two or more times. The first target costs 21 to 33 instructions; the
  last ones cost about 4.
- **The 1,000 puzzles fall into families** (our own labels):

| Family | Puzzles | What the targets look like | Target bits per instruction |
|---|---|---|---|
| Independent | 215 | random numbers in three size classes (about 39, 49 and 59 bits) | 6.3 to 6.5 |
| Mixed | 182 | partly structured, partly random | 6.7 |
| Shared | 264 | small multiples of hidden hubs plus small offsets | 7.9 |
| Multiplicative | 189 | products of hidden seeds with offsets | 6.3 |
| Medium | 134 | 6 to 18 targets of about 25 bits | n/a |
| Tiny | 16 | proved optimal early on (57 instructions in total) | n/a |

- **Random targets cost in proportion to their size.** A private chain of a 40-, 50- or 60-bit target needs 5, 6 or 7
  instructions, because that is the first depth at which a pool of about 240 values can reach a number of that size at all.
  Exhaustive search (section 4) confirmed there is nothing shorter.
- **The register limit matters only on the largest programs.** Programs of 225 or more instructions sit at 46 to 48 live values
  for a large part of their length; three quarters of those registers hold the small "dictionary" values under 24 bits.

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, datetime as dt
INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e6e5e0', '#fcfcfb'
C1, C2, C3, C4, NEUTRAL = '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#b9b8b2'      # fixed categorical order
mpl.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'font.size': 10.5,
    'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'lines.linewidth': 2})
def fig(w=9.5, h=4.2):
    f, ax = plt.subplots(figsize=(w, h)); ax.tick_params(length=0); return f, ax
def hbar(ax, labels, values, color=C1, fmt='{:,.0f}', pad=0.01):
    y = range(len(labels))[::-1]; ax.barh(list(y), values, height=0.62, color=color)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels); ax.grid(axis='y', visible=False)
    span = max(abs(v) for v in values)
    for yi, v in zip(y, values): ax.text(v + (pad * span if v >= 0 else -pad * span), yi, fmt.format(v), va='center', ha='left' if v >= 0 else 'right', color=INK, fontsize=9.5)

## 2. Score over time

We started from the first merge of all public files (138,936) on 19 November. The line is our best local file; the dots are
the scores we submitted. The gold line is the 15th-place score on the public leaderboard at the five moments we saved a snapshot.

In [ ]:
S = [('11-19 16', 138936), ('11-19 20', 138422), ('11-19 22', 138404), ('11-20 00', 138354), ('11-20 02', 138059), ('11-20 04', 137695), ('11-20 06', 136809), ('11-20 08', 136584), ('11-20 10', 136397), ('11-20 12', 134765), ('11-20 14', 134035), ('11-20 16', 133475), ('11-20 18', 133132), ('11-20 20', 132953), ('11-20 22', 132292), ('11-21 00', 131803), ('11-21 02', 131502), ('11-21 04', 131102), ('11-21 06', 130949), ('11-21 08', 130851), ('11-21 10', 130754), ('11-21 12', 130687), ('11-21 14', 130407), ('11-21 16', 130284), ('11-21 18', 130228), ('11-21 20', 130153), ('11-21 22', 130127), ('11-22 00', 130095), ('11-22 02', 130059), ('11-22 04', 130033), ('11-22 06', 130017), ('11-22 08', 129920), ('11-22 10', 129878), ('11-22 16', 129815), ('11-22 18', 129792), ('11-22 20', 129774), ('11-22 22', 129741), ('11-23 00', 129639), ('11-23 02', 129589), ('11-23 04', 129552), ('11-23 06', 129252), ('11-23 08', 129119), ('11-23 10', 129023), ('11-23 12', 128925), ('11-23 14', 128850), ('11-23 16', 128779), ('11-23 18', 128577), ('11-23 20', 128527)]
t = [dt.datetime.strptime('2026-' + a, '%Y-%m-%d %H') for a, _ in S]; s = [b for _, b in S]
t.append(dt.datetime(2026, 11, 23, 23)); s.append(128524)
SUB = [('11-19 22:25', 138406), ('11-20 07:29', 137428), ('11-20 10:48', 136488), ('11-20 13:02', 135370), ('11-20 16:14', 133962),
       ('11-20 22:41', 132569), ('11-21 07:46', 130964), ('11-21 15:37', 130443), ('11-21 23:31', 130132), ('11-22 08:49', 129939),
       ('11-23 07:37', 129287), ('11-23 15:27', 128865), ('11-23 18:30', 128638), ('11-23 20:24', 128560)]
GOLD = [('11-22 16:01', 129639), ('11-22 18:02', 129639), ('11-23 07:38', 128955), ('11-23 18:30', 128365), ('11-23 20:42', 128470)]
p = lambda x: dt.datetime.strptime('2026-' + x, '%Y-%m-%d %H:%M')
f, ax = fig(10, 4.6)
ax.plot(t, s, color=C1, label='Our best file')
ax.plot([p(a) for a, _ in SUB], [b for _, b in SUB], 'o', ms=7, mfc=SURF, mec=C1, mew=2, label='Submitted')
ax.plot([p(a) for a, _ in GOLD], [b for _, b in GOLD], color=C2, marker='s', ms=6, label='Gold cut-off (15th place)')
ax.annotate(f'{s[-1]:,}', (t[-1], s[-1]), xytext=(6, -3), textcoords='offset points', color=INK, fontsize=10)
ax.annotate('138,936', (t[0], s[0]), xytext=(6, 2), textcoords='offset points', color=INK, fontsize=10)
ev = [('11-20 13:14', 'value-graph\nlocal search'), ('11-20 22:40', 'beam move for\nlong chains'), ('11-21 08:35', 'register\nrepair'), ('11-22 22:24', 'faster\nengine'), ('11-23 07:50', 'rebuilds around\nhidden seeds')]
for (a, lab), yy in zip(ev, [138750, 137500, 136250, 138750, 137500]):
    ax.axvline(p(a), color=GRID, lw=1.2, zorder=0); ax.text(p(a) + dt.timedelta(minutes=40), yy, lab, fontsize=8.5, color=INK2, ha='left', va='top')
ax.set_title('Total instructions over time (lower is better)'); ax.set_ylabel('instructions'); ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%d %b')); ax.xaxis.set_major_locator(mpl.dates.DayLocator()); ax.set_xlim(t[0], dt.datetime(2026, 11, 24, 8))
ax.legend(frameon=False, loc='lower left'); plt.show()

Where the 10,400 instructions came from, by family. Shared moved most and independent least: random targets leave little room.

In [ ]:
fam = [('Shared', 264, 35597, 31324), ('Multiplicative', 189, 20052, 18231), ('Mixed', 182, 32766, 30145), ('Medium', 134, 4881, 4560), ('Independent', 215, 45583, 44208)]
f, ax = fig(8.5, 3.2)
hbar(ax, [f'{n}  ({k} puzzles)' for n, k, a, b in fam], [100 * (b / a - 1) for n, k, a, b in fam], fmt='{:.1f}%')
for yi, (n, k, a, b) in zip(range(len(fam))[::-1], fam): ax.text(0.15, yi, f'{a:,} → {b:,}', va='center', color=INK2, fontsize=9)
ax.set_xlim(-14.5, 4); ax.set_title('Change since the first public merge, by family'); ax.set_xlabel('change in instructions'); ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(decimals=0)); plt.show()

## 3. What worked

Ranked roughly by how much each gave us.

**1. A local search on the value graph (most of the gain).** One move: pick a target and remove the instructions only it
uses, or pick a helper value and remove it with everything it alone supports; then rebuild what is missing from *all* remaining
values, exactly, with up to three new instructions. Accept the result if the program is no longer. The first full sweep took
135,370 to 134,723 in an hour. The settings that mattered, each measured on 96 puzzles:

| Setting | Effect on instructions saved |
|---|---|
| Always accept equal-length results (walk the plateau) | 35 → 63 |
| Let single-use values be the root of a move, not just shared helpers | 58 → 83 |
| Among equal rebuilds prefer the one whose new values are small | +18% |
| Remove two helpers at once | +15% |
| Draw each job's configuration from a per-family bandit instead of one fixed setting | +8% |

**2. Merging public programs every day.** Fifteen refreshes; see the chart below. The public frontier is a different lineage of
programs, and it kept finding things our search did not.

**3. A backward beam for long private chains.** Random targets hang on private chains of 5 to 7 instructions that a
three-instruction rebuild cannot touch. A beam that works backwards from the target (subtract, divide, take the remainder
against pool values, with one opening step through any pool value) reproduced or beat existing chains of length 4 / 5 / 6 / 7 in
93 / 85 / 68 / 65% of cases, against 35 / 13 / 0 / 0% without it. It was the first thing that shortened independent puzzles.

**4. A from-scratch constructor with lookahead and a beam.** Useful as a source of new starting programs, not as a solver: its
raw output is 7 to 18% above our best, and about 3% above after 90 seconds of local search. Roughly one fresh start in 25 ends
below our best.

**5. Exact recombination of different programs** (answer-set programming over the pooled values of several programs for the same
puzzle). It carried the first night (138,404 → 137,428) while programs still differed a lot, then dried up completely.

**6. Getting register-blocked finds to fit.** On large programs most shorter graphs peak at 49 to 53 live values. Two things
helped: block moves in the order annealer (four times as many fitted), and a repair walk that changes other parts of the graph
to make room (about one in three converted).

**7. Rebuilding around hidden seeds instead of injecting them.** We could recover planted seeds from the targets for days and
got nothing by adding them to an existing program: a local search cannot re-base a program. Building from scratch on
targets + seeds and then polishing gave 17 wins from 273 builds in the first wave.

**8. Engine speed.** A rewrite of the inner loop made the same search 2 to 5 times faster; yield tracked throughput at roughly
0.4 to 0.5% of the score per doubling.

**9. An audit for silent failures.** A quarter of our alternative starting programs had been skipped for days by a scheduling
call that gave up early, and 84 puzzles had received no search at all after a chain was rearranged. Clearing that backlog alone
gave 60 wins.

In [ ]:
y = [('First sweep of the value-graph search', 50), ('Backward beam on independent puzzles', 15), ('Search under a hard register limit', 10),
     ('Retrying register fits with a heavier annealer', 9), ('Transplant + exact recombination', 7), ('Polishing fresh constructor programs', 5),
     ('Repair walk for register-blocked graphs', 4), ('Recombining plateau variants', 3), ('Seed injection into existing programs', 2),
     ('Rebuilds from scratch around seeds (day 5)', 1.7), ('Polishing public alternates', 1.5), ('Main chains on day 5', 1.0)]
f, ax = fig(9.5, 4.6); hbar(ax, [a for a, _ in y], [b for _, b in y], fmt='{:g}')
ax.set_title('Instructions saved per worker-hour, each method when we measured it'); ax.set_xlabel('instructions per worker-hour'); ax.set_xlim(0, 56); plt.show()

The chart above is the story of the competition in one picture: every method starts strong and decays as the file improves.
By day five everything pays about one instruction per worker-hour, and progress comes from adding new sources, not from tuning.

In [ ]:
r = [559, 932, 39, 307, 35, 3, 212, 6, 23, 72, 29, 245, 6, 11, 113]
f, ax = fig(9.5, 3.4); ax.bar(range(1, 16), r, width=0.62, color=C1); ax.grid(axis='x', visible=False)
for i, v in enumerate(r): ax.text(i + 1, v + 14, f'{v}', ha='center', color=INK, fontsize=9)
ax.set_xticks(range(1, 16)); ax.set_xlabel('public refresh number (19 to 23 November)'); ax.set_ylabel('instructions gained')
ax.set_title('Instructions gained from each merge of new public programs'); plt.show()

## 4. What did not work

Each line is one experiment with its result. Where we ran a positive control it is noted; the others should be read as
"did not work for us in this form".

| Idea | Result |
|---|---|
| Nested rollout policy adaptation on the medium puzzles | 5% worse than the baseline; useful only as material for recombination |
| Annealing through invalid programs (STOKE style) | 0 gains on 134 medium puzzles |
| Sparse number systems (2^a·3^b terms) | 7 to 9 terms per target, more additions than the whole baseline |
| Lattice / small-coefficient relations between targets | about 10 targets in the whole set |
| A fixed toolkit of small constants built up front | worse, twice |
| Re-synthesising a window of 3 to 6 targets with the constructor | 0 shorter in 36 windows (an early "positive" was an accounting error) |
| Keeping 80% of a program's shared values and rebuilding the rest | 10.9% longer: every shared value is tailored to its users |
| A beam that is wide only at the opening | worse than a flat beam; the opening is not where it loses |
| Solving for 4, then 8, then 16, then all targets | 4 to 9% longer: early "waste" is an investment later targets live on |
| Tabu memory on removed values | no effect at short tenure, 30 to 60% less yield at long tenure |
| Annealing with a length allowance instead of plateau walking | half the yield |
| A third helper per move, larger helpers, usefulness tie-breaks | within noise |
| Bridge move: insert one shared value and reroute several values through it | 6,749 applied, 0 shortenings |
| Learned cost-to-go for the constructor's beam | ranks better offline (0.57 against 0.51), no effect on yield at production widths |
| Hub recovery by continued fractions, then hub injection | the common hub of a pair is not the value worth sharing |
| Injecting recovered atoms into the polish | one spectacular case (140 → 117), then 1 win in 125 puzzles |
| Exact recombination across lineages, and over hundreds of plateau states | 0 to 1 wins; most puzzles proved optimal within the pool |
| Exhaustive GPU rebuild of every value with a private chain of 5 or 6 | 2,393 goals, 0 hits (control passed) |
| Exhaustive GPU rebuild of 7-instruction chains with 6 | about 1.5% of goals, and most of those do not fit 48 registers |
| Whole-range search on small-limit puzzles (every value up to 2^30 in GPU memory) | 0 wins on 111 puzzles (controls passed) |
| A much wider constructor beam | see chart: 13 times the time for half a point after polish |
| Pushing lineages apart with an overlap penalty | on shared puzzles 13% above best after polish, against 8.6% without |
| Removing the register limit entirely | only 0.05% shorter: the limit is not what holds programs back |
| Exact shared factors between targets | 30 puzzles have any; direct target-to-target relations cost more than going through the hub |

In [ ]:
f, ax = fig(7.5, 3.8); w = ['6', '24', '96']; raw = [11.2, 10.2, 8.7]; pol = [3.8, 3.7, 3.3]
ax.plot(w, raw, color=C1, marker='o', ms=7, label='Raw constructor output'); ax.plot(w, pol, color=C2, marker='o', ms=7, label='After 60 s of local search')
for x, a, b in zip(w, raw, pol): ax.text(x, a + 0.45, f'+{a}%', ha='center', color=INK, fontsize=9.5); ax.text(x, b + 0.45, f'+{b}%', ha='center', color=INK, fontsize=9.5)
for x, tt in zip(w, ['163 s', '638 s', '2,147 s']): ax.text(x, 0.35, tt + ' per build', ha='center', color=INK2, fontsize=9)
ax.set_ylim(0, 13); ax.margins(x=0.18); ax.set_xlabel('beam width'); ax.set_ylabel('length above our best'); ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(decimals=0))
ax.set_title('A wider constructor beam barely survives the polish (12 puzzles)'); ax.legend(frameon=False, loc='center right'); plt.show()

## 5. Insights

**Private chains sit at an information floor.** With a pool of n values, c new instructions reach on the order of n^(c+1) numbers.
For n around 240 that first covers 40-bit numbers at c = 5, 50-bit at 6 and 60-bit at 7, which is exactly the chain length good
programs have. The distribution of chain lengths over independent targets:

In [ ]:
f, ax = fig(7.5, 3.3); x = ['1 to 3', '4', '5', '6', '7', '8']; v = [54, 18, 16, 8, 3.5, 0.8]
ax.bar(x, v, width=0.62, color=C1); ax.grid(axis='x', visible=False)
for i, a in enumerate(v): ax.text(i, a + 1.2, f'{a:g}%', ha='center', color=INK, fontsize=9.5)
ax.set_ylim(0, 62); ax.set_xlabel('instructions used only by that target'); ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(decimals=0))
ax.set_title('Share of independent targets by length of their private chain'); plt.show()

**Two serious pipelines land several percent apart on the same puzzle, in both directions.** When another team's programs for 258
puzzles became public we compared them one by one with ours. Theirs were 0.7% longer in total, yet shorter on 49 puzzles, by up
to 12 instructions; ours were shorter by up to 17. Their winning programs shared almost no intermediate values with ours. Neither
side is near the floor on mixed and independent puzzles, and the best of several deep lineages is worth more than a deeper polish
of one.

In [ ]:
fams = ['Independent (130)', 'Mixed (109)', 'Shared (18)']; they = [17, 30, 2]; tie = [33, 16, 1]; we = [80, 63, 15]
f, ax = fig(8.5, 2.9); y = [2, 1, 0]; tot = [a + b + c for a, b, c in zip(they, tie, we)]
sh = lambda v: [100 * a / t for a, t in zip(v, tot)]
ax.barh(y, sh(they), height=0.6, color=C2, label='Their program shorter', edgecolor=SURF, linewidth=2)
ax.barh(y, sh(tie), left=sh(they), height=0.6, color=NEUTRAL, label='Same length', edgecolor=SURF, linewidth=2)
ax.barh(y, sh(we), left=[a + b for a, b in zip(sh(they), sh(tie))], height=0.6, color=C1, label='Ours shorter', edgecolor=SURF, linewidth=2)
for yi, a, b, c, t_ in zip(y, they, tie, we, tot):
    ax.text(100 * a / t_ / 2, yi, str(a), ha='center', va='center', color='white', fontsize=10); ax.text(100 * (a + b / 2) / t_, yi, str(b), ha='center', va='center', color=INK, fontsize=10)
    ax.text(100 * (a + b + c / 2) / t_, yi, str(c), ha='center', va='center', color='white', fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(fams); ax.grid(visible=False); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(decimals=0))
ax.set_title('Two pipelines on the same 257 puzzles: who has the shorter program'); ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.18)); plt.show()

**How alike are two programs for the same puzzle?** The share of intermediate values that a near-best public program has in
common with ours. On independent puzzles every public program is a relative of every other; on mixed and multiplicative puzzles
any two builds are almost disjoint, so new lineages are cheap there and it pays to keep all of them.

In [ ]:
f, ax = fig(7.5, 3.0); hbar(ax, ['Independent', 'Shared', 'Medium', 'Mixed', 'Multiplicative'], [52, 36, 18, 10, 9], fmt='{:.0f}%')
ax.set_xlim(0, 60); ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(decimals=0)); ax.set_xlabel('median share of intermediate values in common')
ax.set_title('Overlap between a near-best public program and ours'); plt.show()

**A local search cannot re-base a program.** On generated puzzles with a known planted solution, a program that starts without
the planted seeds stays about 10% too long however long it is polished (7% after ten minutes instead of ninety seconds). The same
program built from scratch around the seeds reaches the planted length. This is why injection experiments failed for days while
rebuilds worked on the first try.

**Raw length is a poor guide to polished length.** Across fresh builds the correlation is 0.44. Picking "the best of three
constructor seeds" before polishing throws away two programs that are just as likely to win.

**Every method decays; new sources are what move the score.** Tuning gave percent-level changes in yield. Each new source of
different programs (the constructor, public merges, seed rebuilds, another team's lineage) gave a step.

**Negative results need a positive control.** At least four of our "this is exhausted" conclusions were wrong: one because a
pool-building script had a bug, one because the injection step (not the recovery) was failing, one from an accounting error in
our own test, one because a queue silently dropped its inputs. We now refuse to close a line until the same pipeline has been
shown to recover a planted or known gain.

**On hardware.** The work ran on one 16-core workstation with one GPU. Running twice as many jobs as threads costs almost
nothing in throughput (98% of time in user code); it only halves what a wall-clock budget buys. The GPU is worth roughly 7 to 15
cores on the lookup-heavy steps of this problem, and much more on bulk enumeration with an early filter.

## 6. The submission

The 1,000 programs are stored below in a compressed text form. The cell decodes them, replays every program against
`puzzles.csv` with the competition's rules (48 registers, exact division, positive results within the value limit, every target
produced), and writes `submission.csv`. Expected total: **128,524**.

In [ ]:
import base64, lzma, csv, json, glob
PROGRAMS_XZ_B64 = "..."
OPS = {'a': '+', 's': '-', 'm': '*', 'd': '/'}
programs = {}
for line in lzma.decompress(base64.b64decode(PROGRAMS_XZ_B64)).decode().split('\n'):
    pid, *ins = line.split(' ')
    programs[pid] = [[OPS[x[0]], *map(int, x[1:].split(','))] for x in ins]

def validate(pz, prog):
    S, T, L = json.loads(pz['starting_values']), set(json.loads(pz['targets'])), int(pz['value_limit'])
    R = [None] * 48; R[:len(S)] = S; need = T - set(S)
    for op, i, j, d in prog:
        a, b = R[i], R[j]
        if a is None or b is None: return False
        if op == '+': v = a + b
        elif op == '-': v = a - b
        elif op == '*': v = a * b
        else:
            if a % b: return False
            v = a // b
        if not 1 <= v <= L: return False
        R[d] = v; need.discard(v)
    return not need and len(prog) <= 4096

path = (glob.glob('/kaggle/input/**/puzzles.csv', recursive=True) + glob.glob('puzzles.csv'))[0]
puzzles = {r['id']: r for r in csv.DictReader(open(path))}
scored = sorted(p for p in puzzles if p in programs)
bad = [p for p in scored if not validate(puzzles[p], programs[p])]
total = sum(len(programs[p]) for p in scored)
print(f'{len(scored)} puzzles, {len(bad)} invalid, total instructions {total:,}')
assert len(scored) == 1000 and not bad and total == 128524
with open('submission.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['id', 'program'])
    for p in scored: w.writerow([p, json.dumps(programs[p], separators=(',', ':'))])
print('wrote submission.csv')

## How this was made

Everything here was built in [Claude Code](https://claude.com/claude-code), Anthropic's coding agent, running on one workstation:
the solvers (C++ search engines, the constructor, the GPU searches), the pipeline that keeps them running, all 177 logged
experiments, the analysis and this notebook. No code was written by hand. My part was steering: suggesting ideas to test
(several of the rows in section 4 started as my suggestions), asking where the weak points were, deciding priorities and choosing
when to submit.

Two habits made that work. Every experiment was logged as it happened with a question, what was done, the result and a
conclusion, so that later sessions could build on earlier ones. And after an audit found several silent failures, no line of
attack was closed on a negative result without a positive control.

## Credits: where the merged programs came from

This file stands on public work, and about a quarter of our total gain came straight from it: across fifteen merges we took
**1,044 programs worth 2,590 instructions** from public notebooks (of the 10,400 gained since the first merge). The rest
came from our own search running on top.

**How the attribution was done.** For every merge we compared our file before and after, and matched each program we took,
character for character, against every public notebook output we had downloaded (170 outputs). A program is credited to the
earliest-dated output that contains it. Two caveats: most public notebooks are themselves merges, so the same program usually
sits in several of them and the true originator may be upstream of the one named here; and we only hold each notebook's latest
output, so a program that a notebook later replaced is credited to whoever still carried it. If we have credited your work to
someone else, tell us and we will fix it.

**Programs taken in the fifteen merges, by author**

| Author | Notebooks | Programs taken | Instructions gained |
|---|---|---|---|
| [royalacecat](https://www.kaggle.com/royalacecat) | elf-calculator-residual-cut-search, santa-2026-crossover-stage, santa-2026-public-frontier | 365 | 1091 |
| [kozykappa](https://www.kaggle.com/kozykappa) | santa-lb | 134 | 303 |
| [wocenmma](https://www.kaggle.com/wocenmma) | santa-2026-134753-verified-arithmetic-programs | 113 | 243 |
| [goodpjw2008](https://www.kaggle.com/goodpjw2008) | santa-2026-dag-lns-depth-5-search, santa-2026-lns-cp-sat-recombine, santa-2026-source-aware-frontier | 77 | 217 |
| [nevska](https://www.kaggle.com/nevska) | santa-2026-explorer | 35 | 138 |
| [kindasomethin](https://www.kaggle.com/kindasomethin) | santa-2026-exact-portfolio-frontier | 53 | 119 |
| [tkm2261](https://www.kaggle.com/tkm2261) | sorry-for-my-leakage-it-s-all-my-ai-s-fault | 48 | 105 |
| [ishankarnick](https://www.kaggle.com/ishankarnick) | santa-2026-frontier-programs | 31 | 66 |
| [hidsaito](https://www.kaggle.com/hidsaito) | santa2026-a005 | 48 | 60 |
| [atakanaldemir](https://www.kaggle.com/atakanaldemir) | santa-2026-joint-recipes-for-large-puzzles | 36 | 49 |
| [bikrambasnet1](https://www.kaggle.com/bikrambasnet1) | santa-2026-arithmetic-graph-rewrites | 17 | 26 |
| [lucifer19](https://www.kaggle.com/lucifer19) | santa-2026-the-elf-calculator-x-ray | 7 | 22 |
| [pardheev](https://www.kaggle.com/pardheev) | 132-715-best-public-score-santa-2026, 132-904-best-public-score-santa-2026, 132-948-best-public-score-santa-2026 | 15 | 20 |
| [itbert](https://www.kaggle.com/itbert) | santa-2k26 | 3 | 16 |
| [avazaku](https://www.kaggle.com/avazaku) | elf-calc-paired-helpers-and-target-scheduling, elf-calc-signed-regions-and-register-scheduling | 5 | 15 |
| [menes97](https://www.kaggle.com/menes97) | santa-2026-deep-cone-rebuilds | 5 | 15 |
| [avioon](https://www.kaggle.com/avioon) | santa-2026-public-merge | 4 | 12 |
| [lynnsakurai](https://www.kaggle.com/lynnsakurai) | santa-gauss-and-the-r-a-a-p | 7 | 10 |
| [asteriospanoras](https://www.kaggle.com/asteriospanoras) | santa-2026-best-public-score-dataset-fails | 7 | 9 |
| [mizeroluckygall](https://www.kaggle.com/mizeroluckygall) | elf-calculator-best-of-public-merge-polish | 6 | 9 |
| [wangjakie](https://www.kaggle.com/wangjakie) | santa-2026-arithmetic-recipe-pool-and-milp | 6 | 8 |
| [pablodiegoo](https://www.kaggle.com/pablodiegoo) | santa-2026-deterministic-atlas-solver | 4 | 6 |
| [santiagozazo](https://www.kaggle.com/santiagozazo) | santa-2026-137-792-cp-sat-multi-bank-fuser | 1 | 5 |
| [vigneshsankar1962](https://www.kaggle.com/vigneshsankar1962) | santa-2026-132k-sota-frontier-synthesis-portfolio | 2 | 5 |
| [muhakabartay](https://www.kaggle.com/muhakabartay) | santa-2026-strict-verifier-public-merge-anatomy | 2 | 4 |
| [kmmmvs](https://www.kaggle.com/kmmmvs) | analysis-santa-2026-starter-notebook, santa-2026-verified-best-of-merge | 3 | 3 |
| [hanifnoerrofiq](https://www.kaggle.com/hanifnoerrofiq) | santa-2026-elf-calculator-verified-portfolio | 3 | 3 |
| [rishabhbafnaiiitd](https://www.kaggle.com/rishabhbafnaiiitd) | arithmetic-dag-synthesis-and-register-allocation | 1 | 3 |
| [amanatar](https://www.kaggle.com/amanatar) | santa-2026-fast-validator-eda-sota-baseline | 1 | 3 |
| [romanrozen](https://www.kaggle.com/romanrozen) | santa-2026-icefire-strong-start | 2 | 2 |
| [ozertuu](https://www.kaggle.com/ozertuu) | santa-2026-starter-notebook | 2 | 2 |
| [prvsiyan](https://www.kaggle.com/prvsiyan) | santa-is-not-happy | 1 | 1 |

**The starting file.** Our first file (138,936 on 19 November) was a pure per-puzzle merge of the public notebooks of that day.
By first appearance its 1,000 programs came from: [skagumire](https://www.kaggle.com/skagumire) (355), [lightningv08](https://www.kaggle.com/lightningv08) (320), [ozertuu](https://www.kaggle.com/ozertuu) (98), [konbu17](https://www.kaggle.com/konbu17) (73), [avazaku](https://www.kaggle.com/avazaku) (65), [muchat007](https://www.kaggle.com/muchat007) (47), [b05204047elias](https://www.kaggle.com/b05204047elias) (18), [malyshevdanil](https://www.kaggle.com/malyshevdanil) (7), [mizeroluckygall](https://www.kaggle.com/mizeroluckygall) (5), [tonylica](https://www.kaggle.com/tonylica) (4), [linjunren1016](https://www.kaggle.com/linjunren1016) (2), [rishabhbafnaiiitd](https://www.kaggle.com/rishabhbafnaiiitd) (1), [diftaradityapratama](https://www.kaggle.com/diftaradityapratama) (1), [sheroleg](https://www.kaggle.com/sheroleg) (1).

**Still verbatim in the submission below.** 241 of the 1,000 programs in today's file are exactly a public program; the other
759 have been changed by our search since they were merged. The verbatim ones were first seen in notebooks by: [royalacecat](https://www.kaggle.com/royalacecat) (94), [tkm2261](https://www.kaggle.com/tkm2261) (32), [ishankarnick](https://www.kaggle.com/ishankarnick) (19), [goodpjw2008](https://www.kaggle.com/goodpjw2008) (14), [pardheev](https://www.kaggle.com/pardheev) (8), [lightningv08](https://www.kaggle.com/lightningv08) (8), [malyshevdanil](https://www.kaggle.com/malyshevdanil) (7), [lynnsakurai](https://www.kaggle.com/lynnsakurai) (5), [mizeroluckygall](https://www.kaggle.com/mizeroluckygall) (5), [atakanaldemir](https://www.kaggle.com/atakanaldemir) (5).

**Also used**

- The register-allocator notebook by [alexchilton](https://www.kaggle.com/alexchilton): in several merges this was the file we read the winning programs
  from, and we learned from its write-up on register allocation.
- The fairness patch by [tkm2261](https://www.kaggle.com/tkm2261) and the query datasets published by [nagiss](https://www.kaggle.com/nagiss): 48 programs came from the patch,
  and 6 more were rebuilt by us from the query pools in the datasets.
- Ideas from public write-ups that we adopted or tested: transplanting the shared values of a good program, value-graph
  large-neighbourhood search with plateau walking, CP-SAT and answer-set recombination, deep cone rebuilds on the GPU.

Thank you to everyone who shares. If you build on this, the most useful thing you can add is a lineage of your own: on mixed and
independent puzzles two independent searches rarely agree, and the union is what wins.
